In [1]:
import requests
from bs4 import BeautifulSoup

## Request html from USGS website (Europe and Central Euroasia)
usgs_region_url = "https://www.usgs.gov/centers/national-minerals-information-center/asia-and-pacific"
usgs_region_response = requests.get(url = usgs_region_url,
                                    headers= {
                                        'User-Agent': "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/149.0.0.0 Safari/537.36 Edg/149.0.0.0"
                                    })

usgs_region_response.encoding = 'utf-8'
usgs_region_soup = BeautifulSoup(usgs_region_response.content, features='html.parser')

In [2]:
print(usgs_region_soup.prettify())

<!DOCTYPE html>
<html dir="ltr" lang="en" prefix="og: https://ogp.me/ns#">
 <head>
  <!-- Google Tag Manager -->
  <script>
   (function(w,d,s,l,i){w[l]=w[l]||[];w[l].push({'gtm.start':
        new Date().getTime(),event:'gtm.js'});var f=d.getElementsByTagName(s)[0],
      j=d.createElement(s),dl=l!='dataLayer'?'&l='+l:'';j.async=true;j.src=
      'https://www.googletagmanager.com/gtm.js?id='+i+dl;f.parentNode.insertBefore(j,f);
    })(window,document,'script','dataLayer', 'GTM-KWRSKS');
  </script>
  <!-- End Google Tag Manager -->
  <meta charset="utf-8"/>
  <meta content="Reports on countries' mineral industries, including statistics on mineral production and trade, government policies, industry structure, and commodity sector developments." name="description"/>
  <meta content="China,Indonesia,mongolia" name="keywords"/>
  <link href="https://www.usgs.gov/centers/national-minerals-information-center/asia-and-pacific" rel="canonical"/>
  <meta content="USGS" property="og:site_name"/

In [ ]:
sections_list = usgs_region_soup.find_all('h2')
country_reports_section = [section for section in sections_list if 'country reports' in section.text.lower() and 'multi' not in section.text.lower()][0]
country_reports_section_iterator = country_reports_section.next_siblings
## Module: Split list into slices, given the conditions cutpoints should satisfy
split_output = []
current_slice = []
for label in country_reports_section_iterator:
    if label.name == 'h5': # Conditions cutpoints should satisfy (identifying condition of cutpoints)
        if current_slice:
            split_output.append(current_slice)
            current_slice = [label]
    else:
        current_slice.append(label)
split_output.append(current_slice)
## --------------------------------------------------------
split_output = split_output[1:] # Remove chapeau

import pandas as pd
import sys; sys.path.append("../../modules/retrieve_url/")
import RetrieveCountrySite as rcs
import time
import random

reports_dict = {}
tables_dict = {}
for slice in split_output:
    country = slice[0].get_text(strip=True)

    reports_index = [index for index, head in enumerate(slice) if 'Report' in list(head.children)[0]]
    tables_index = [index for index, head in enumerate(slice) if 'Table' in list(head.children)[0]]
    reports_index = reports_index[0] if len(reports_index) != 0 else None
    tables_index = tables_index[0] if len(tables_index) != 0 else None

    if reports_index is not None:
        reports_dict[country] = {}
        reports_label_list = slice[reports_index].find_all('a')
        for report_label in reports_label_list:
            year = report_label.get_text(strip=True)
            report_url = report_label.get('href', default='')
            if report_url.endswith('.pdf') and report_url.startswith(('http','https')):
                reports_dict[country][year] = report_url
    if tables_index is not None:
        tables_dict[country] = {}
        tables_label_list = slice[tables_index].find_all('a')
        for table_lable in tables_label_list:
            year = table_lable.get_text(strip=True)
            table_url = table_lable.get('href', default='')
            if table_url.endswith(('.xls','.xlsx')) and table_url.startswith(('http','https')) and len(year) != 0:
                tables_dict[country][year] = table_url
    if reports_index is None and tables_index is None and slice[0].find('a') is not None:
        for label in slice[0].find_all('a'): # IMPORTANT DEBUG: We mistakenly wrote .find('a') instead of .find_all('a'), leaving it possible to return None
            attempt = label.get('href') # Try to get 'href' from label (None is returned if the field does not exist in the current label)
            if attempt is not None:
                country_url = attempt
        print(f"Visiting homepage of [{country}]..."); start_time = time.perf_counter()
        try: # IMPORTANT DEBUG: 
            reports_dict.update(rcs.reports_tables_url(country_name = country, country_url = country_url)[0])
            tables_dict.update(rcs.reports_tables_url(country_name = country, country_url = country_url)[1])
            print(f"Scraping completed! Time consumed: {time.perf_counter()-start_time:.2f}s")
        except requests.exceptions.Timeout:
            print("Failed to scrape! Error: Time out")
        sleep_time = 3 + random.uniform(0,3); time.sleep(sleep_time); print(f"Snap ended. Time consumed: {sleep_time:.2f}s")

Visiting homepage of [Afghanistan]...
Scraping completed! Time consumed: 3.12s
Snap ended. Time consumed: 4.47s
Visiting homepage of [Bangladesh]...
Scraping completed! Time consumed: 3.58s
Snap ended. Time consumed: 3.75s
Visiting homepage of [Bhutan]...
Scraping completed! Time consumed: 3.27s
Snap ended. Time consumed: 3.13s
Visiting homepage of [Brunei]...
Scraping completed! Time consumed: 3.42s
Snap ended. Time consumed: 5.21s
Visiting homepage of [Burma(Myanmar)]...
Scraping completed! Time consumed: 4.88s
Snap ended. Time consumed: 4.09s
Visiting homepage of [Cambodia]...
Scraping completed! Time consumed: 6.26s
Snap ended. Time consumed: 4.12s
Visiting homepage of [Indonesia]...
Scraping completed! Time consumed: 7.59s
Snap ended. Time consumed: 5.51s
Visiting homepage of [Korea, North]...
Scraping completed! Time consumed: 4.72s
Snap ended. Time consumed: 5.41s
Visiting homepage of [Laos]...
Scraping completed! Time consumed: 5.03s
Snap ended. Time consumed: 3.35s
Visiting ho

In [4]:
tables_dict

{'Afghanistan': {'2003': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/2003/afmyb03.xls',
  '2008': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/2008/myb3-2008-af.xls',
  '2009': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/2009/myb3-2009-af.xls',
  '2010': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/2010/myb3-2010-af.xls',
  '2011': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/2011/myb3-2011-af.xls',
  '2012': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/2012/myb3-2012-af.xls',
  '2013': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/mineral-pubs/country/2013/myb3-2013-af.xls',
  '2014': 'https://d9-wret.s3.us-west-2.amazonaws.com/assets/palladium/production/s3fs-public/media/f

In [8]:
# FOR DEBUGGING
slice = split_output[0]
print(slice)

[<h5><a class="ck-anchor" id="afg"></a><a class="ck-anchor" id="afg"> </a><a href="https://usgs.gov/centers/national-minerals-information-center/afghanistan"><strong>Afghanistan</strong></a></h5>, <p> </p>]


In [6]:
import re
tables_df = pd.DataFrame(tables_dict).reset_index(names=['year'])
tables_df['year'] = tables_df['year'].str.replace(r'[a-zA-Z\s]', '', regex=True).str.strip('-').replace('2017-18','2018',regex=False).replace('2020-21','2021',regex=False)
tables_df = tables_df.melt(id_vars='year', var_name='country', value_name='table_url').sort_values(by=['country','year']).dropna()
tables_df.to_csv("../../data/original_url/tables_new/usgs_asia_tables.csv", index=False)

In [7]:
import re
reports_df = pd.DataFrame(reports_dict).reset_index(names=['year'])
reports_df['year'] = reports_df['year'].str.replace(r'[a-zA-Z\s]', '', regex=True).str.strip('-').replace('2017-18','2018',regex=False).replace('2020-21','2021',regex=False)
reports_df = reports_df.melt(id_vars='year', var_name='country', value_name='report_url').sort_values(by=['country','year']).dropna()
reports_df.to_csv("../../data/original_url/reports_new/usgs_asia_reports.csv", index=False)